In [1]:
import pandas as pd
import numpy as np
import os, csv, re
import matplotlib.pyplot as plt
from pathlib import Path
plt.close("all")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.0f}'.format) # Turn off scientific notation
pd.set_option('display.float_format', '{:.2f}'.format) # Force display floats with 2 decimal globally

def format_headers(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace(r"[./ >#:?*\-]", "_", regex=True)
        .str.replace(r"[()']", "", regex=True)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.lower()
        .str.replace(r"_+", "_", regex=True)
        .str.strip()
        .str.strip("_")
    )
    return df


from decimal import Decimal, ROUND_HALF_UP

In [2]:
# Selects which data set we want to analyse

# subject="math"
subject="reading"
subject_title = subject.title()
subject_title

'Reading'

In [3]:
# Opens the file that we want to use and read
df = pd.read_csv(f"exports/2526_diagnostic_results_{subject}_CONFIDENTIAL.csv")
df = format_headers(df)
df.columns

Index(['student_id', 'student_last_name', 'student_first_name',
       'student_grade', 'school', 'completion_date', 'norming_window',
       'most_recent_diagnostic_ytd_y_n', 'overall_relative_placement',
       'percent_progress_to_annual_typical_growth'],
      dtype='str')

In [4]:
# Gives the value count of "Rush Flag"

# df["Rush Flag"].value_counts(dropna=False)

In [5]:
# Assigns where "Rush Flag" is "Red Rush Flag" back to our data frame

# df=df[df["Rush Flag"]=="Red Rush Flag"]

In [6]:
# reduces data frame to Rush Flag and Quantile Range

# df=df[["Rush Flag", "Quantile Range"]]

In [7]:
# Reduce dataframe to what we need

df = df[["student_id", "student_grade", "school", "completion_date", "norming_window", "most_recent_diagnostic_ytd_y_n",
          "overall_relative_placement", "percent_progress_to_annual_typical_growth"]]

In [8]:
# Checks the value counts of our data

# for col in df.columns:
    
    
#     print(df[col].value_counts(dropna = False))
#     print()

In [9]:
# Checks our student_grade and norming_window, used to see if 9,10,11,12 cancels out

# pd.crosstab(
#     df["student_grade"],
#     df["norming_window"]
# )

In [10]:
# Creates a copy of our data frame

aoy_df = df.copy()

In [11]:
# If we have extra grades showing it gets rid of those extras

aoy_df = aoy_df[aoy_df["student_grade"].isin(["K","1","2","3","4","5","6","7","8"])]

In [12]:
# Creates met_performance column with base value 0 and assigns a 1 to all students that have met performance

aoy_df["met_performance"] = 0
aoy_df.loc[ (aoy_df["overall_relative_placement"] == "Mid or Above Grade Level") | 
    (aoy_df["overall_relative_placement"] == "Early On Grade Level"), "met_performance"] = 1

In [13]:
# changes completion date from a string to a date

aoy_df["completion_date"] = pd.to_datetime(aoy_df["completion_date"])

In [14]:
# sorts our columns so later we can get rid of duplicates

# aoy_df = aoy_df.sort_values(["completion_date","most_recent_diagnostic_ytd_y_n"], ascending = [False,False])
# aoy_df

In [15]:
# Gets rid of duplicates based on student_id and norming_window

aoy_df = aoy_df.drop_duplicates(subset = ["student_id", "norming_window"], keep = "first")

In [16]:
# Fills in blanks with 0's in percent_progress_to_annual_typical_growth  

aoy_df = aoy_df.fillna({"percent_progress_to_annual_typical_growth" : 0})

In [17]:
# Fills in blanks with 0's in percent_progress_to_annual_typical_growth  

# aoy_df["percent_progress_to_annual_typical_growth"] = aoy_df["percent_progress_to_annual_typical_growth"].fillna(0)

In [18]:
# Since NA is gone changes the type to an integer

aoy_df["percent_progress_to_annual_typical_growth"] =  aoy_df["percent_progress_to_annual_typical_growth"].astype(int) 

In [19]:
# Creates a copy of just the end of year data

eoy_df = aoy_df[aoy_df["norming_window"] == "Spring (March 2 - End of Year)"].copy()

In [20]:
# Gets the value counts of student_grade

# eoy_df["student_grade"].value_counts(dropna = False)

In [21]:
# Creates a copy of just the beginning of year data

boy_df = aoy_df[aoy_df["norming_window"] == "Fall (Beginning of Year - November 15)"].copy()

In [22]:
# Creates column in eoy_df, has_boy, and assigns a 1 if the student_id is in both eoy and boy and assigns a 0 if not

boy_students = set(boy_df["student_id"])

eoy_df["has_boy"] = np.where(eoy_df["student_id"].isin(boy_students), 1, 0)
# eoy_df["has_boy"].value_counts(dropna=False)


In [23]:
# If student_id is not in eoy and boy sets to blank. Then assigns values of 1 and 0 if the students growth is >= 100

eoy_df['met_growth'] = np.where(
    eoy_df['has_boy'] == 0,
    np.nan,
    np.where(eoy_df['percent_progress_to_annual_typical_growth'] >= 100, 1, 0)
)
# eoy_df['met_growth'].value_counts(dropna=False)

In [24]:
# Creates a pivot table to sum and count the data

table = pd.pivot_table( eoy_df, values = ["met_performance", "met_growth"], index = ["school", "student_grade"], 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})

In [25]:
# Creates another pivot table with the same data but combined for the whole district

table2 = pd.pivot_table(eoy_df, values = ["met_performance", "met_growth"], index = "student_grade", 
                        aggfunc = {"met_growth" : ["count","sum"], "met_performance" : ["count","sum"]})
table2["school"] = "Summit Valley School District"
table2 = table2.set_index("school",append=True)
table2 = table2.swaplevel(0,1)

In [26]:
# Calculates the met_growth percent for both tables

table["met_growth" , "percent"] = table["met_growth" , "sum"] / table["met_growth" , "count"]
table2["met_growth" , "percent"] = table2["met_growth" , "sum"] / table2["met_growth" , "count"]

In [27]:
# Calculates the met_performance percent for both tables

table["met_performance", "percent"] = table["met_performance", "sum"] / table["met_performance", "count"]
table2["met_performance", "percent"] = table2["met_performance", "sum"] / table2["met_performance", "count"]

In [28]:
# Puts the percent columns witht he already existing met_growth and met_performance columns

table = table.sort_index(axis=1)
table2 = table2.sort_index(axis=1)

In [29]:
# Merges together table and table2

table = pd.concat([table, table2])

In [30]:
# Assigns school name to all the columns

table = table.reset_index()

In [31]:
# Combines the stacked labels into 1. So met_growth and met_performance got divided into their percent, sum, and count

table.columns = [
    '_'.join(map(str, col)).strip()
    for col in table.columns
]
table.columns = table.columns.str.rstrip('_')

In [32]:
# Creates a folder

output_dir = Path("intermediate")
output_dir.mkdir(parents=True, exist_ok=True)

In [33]:
# Exports the table

table.to_csv(f"intermediate/01_CleanTest_{subject}.csv", index=False)
print("Done")

Done
